### Esercizi SQL Esteso

Sono date le seguenti tabelle:

CIRCOLO (IDCircolo, NomeCircolo, Città, Provincia, Regione, SalaRitrovo, Docce, Bar, Spogliatoi)

MAESTRO (IDMaestro, NomeMaestro, Nazionalità, FasciaEtà)

TEMPO-DATA (IDTempoD, Data, Mese, 2-Mesi, 3-Mesi, 6-Mesi, Anno)

TEMPO-ORA (IDTempoO, FasciaOraria)

JUNK-LEZIONE (IDJL, TipologiaLezione, LivelloLezione, ModalitàPagamento)

LEZIONI-TENNIS (IDCircolo, IDMaestro, IDTempoD, IDTempoO, IDJL, NumOre, NumAllievi, Importo)



1. Per ciascun maestro, livello di lezione e per ogni semestre (attributo 6-Mesi), visualizzare:

- il numero medio di ore di lezione mensili

- il numero complessivo di ore di lezione, separatamente per ogni anno

- la percentuale del numero di ore di lezione rispetto al numero complessivo di ore di lezione considerando tutti i maestri della stessa nazionalità 

    Assegnare ad ogni record una posizione in una classifica, calcolata in base al numero complessivo di ore di lezione, separatamente per nazionalità del maestro.

In [ ]:
SELECT M.IDMaestro, JL.LivelloLezione, TD."6-Mesi",
    SUM(LT.NumOre) / COUNT(DISTINCT TD.Mese),
    SUM(SUM(LT.NumOre)) OVER (
        PARTITION BY TD.Anno
    ),
    100 * SUM(LT.NumOre) / SUM(SUM(LT.NumOre)) OVER (
        PARTITION BY M.Nazionalità, TD."6-Mesi", JL.LivelloLezione
    ),
    RANK() OVER (
        PARTITION BY M.Nazionalità
        ORDER BY (SUM(SUM(LT.NumOre))) DESC
    )
FROM "LEZIONI-TENNIS" LT
JOIN "TEMPO-DATA" TD ON TD.IDTempoD = LT.IDTempoD
JOIN MAESTRO M ON M.IDMaestro = LT.IDMaestro
JOIN "JUNK-LEZIONE" JL ON JL.IDJL = LT.IDJL
GROUP BY M.IDMaestro, M.NomeMaestro, M.Nazionalità, JL.LivelloLezione, TD."6-Mesi", TD.Anno ; 

2. Per ciascuna fascia oraria, mese e città del circolo, visualizzare:

- importo medio per allievo

- numero medio di ore di lezione per allievo, separatamente per trimestre (attributo 3-Mesi)

- rapporto tra il numero di allievi e il numero complessivo di allievi considerando tutti i circoli siti nella stessa provincia

    Effettuare le analisi separatamente per modalità di pagamento.

In [ ]:
SELECT TO.FasciaOraria, TD.Mese, C.Città,
    1.0 * SUM(LT.Importo) / SUM(LT.NumAllievi),
    1.0 * SUM(SUM(LT.NumOre)) OVER (
           PARTITION BY C.Città, TOR.FasciaOraria, TD."3-Mesi", JL.ModalitàPagamento
        ) / SUM(SUM(LT.NumAllievi)) OVER (
           PARTITION BY C.Città, TOR.FasciaOraria, TD."3-Mesi", JL.ModalitàPagamento
        ),
    SUM(LT.NumAllievi) / SUM(SUM(LT.NumAllievi)) OVER (
        PARTITION BY C.Provincia, TD.Mese, TOR.FasciaOraria, JL.ModalitàPagamento
    )
FROM "LEZIONI-TENNIS" LT
JOIN "TEMPO-DATA" TD ON TD.IDTempoD = LT.IDTempoD
JOIN CIRCOLO C ON C.IDCircolo = LT.IDCircolo
JOIN "TEMPO-ORA" TO ON TO.IDTempoO = LT.IDTempoO
JOIN "JUNK-LEZIONE" JL ON JL.IDJL = LT.IDJL
GROUP BY TO.FasciaOraria, TD.Mese, TD."3-Mesi", C.Città, C.Provincia, JL.ModalitàPagamento ;

3. Per ciascun circolo dotato di docce e sala ritrovo, e per ogni bimestre (attributo 2-Mesi), visualizzare:

- il numero medio di ore di lezione per allievo e l’importo medio per ora di lezione

- il rapporto tra le ore di lezione del circolo e le ore di lezione complessive di tutti i circoli siti nella stessa regione

- il numero cumulativo di ore di lezione al passare dei bimestri, separatamente per anno

    Effettuare le analisi per ogni tipologia di lezione.

In [ ]:
SELECT C.IDCircolo, C.NomeCircolo, TD."2-Mesi", JL.TipologiaLezione,
    1.0 * SUM(LT.NumOre) / SUM(LT.NumAllievi),
    1.0 * SUM(LT.Importo) / SUM(LT.NumOre),
    1.0 * SUM(LT.NumOre) / SUM(SUM(LT.NumOre)) OVER(
        PARTITION BY C.Regione, TD."2-Mesi", JL.TipologiaLezione
    ),
    SUM(SUM(LT.NumOre)) OVER(
        PARTITION BY TD.Anno, C.IDCircolo, JL.TipologiaLezione
        ORDER BY TD."2-Mesi" ASC
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    )
FROM "LEZIONI-TENNIS" LT
JOIN "TEMPO-DATA" TD ON TD.IDTempoD = LT.IDTempoD
JOIN CIRCOLO C ON C.IDCircolo = LT.IDCircolo
JOIN "JUNK-LEZIONE" JL ON JL.IDJL = LT.IDJL
WHERE C.Docce = 1 AND C.SalaRitrovo = 1
GROUP BY C.IDCircolo, C.NomeCircolo, C.Regione, TD."2-Mesi", TD.Anno, JL.TipologiaLezione;

Dato lo schema logico precedente, considerare le seguenti query di interesse: 

a. Per i circoli con disponibilità di docce e spogliatoi ma non di bar, visualizzare il valore cumulativo dell’importo delle lezioni al trascorrere dei trimestri (attributo 3-Mesi), separatamente per anno, modalità pagamento (attributo ModalitàPagamento) e regione del circolo. 

b. Separatamente per bimestre, tipologia di lezione e livello di lezione, visualizzare la percentuale dell’importo rispetto al complessivo dell’importo per anno e il numero medio di allievi per ora di lezione, considerando solo circoli della città di Torino. 

c. Per le lezioni di tipologia ‘collettiva’ (attributo TipologiaLezione), separatamente per anno e provincia del circolo, visualizzare l’importo totale e l’importo medio per allievo.


Dato lo schema logico precedente, si svolgano le seguenti attività

1. Definire una vista materializzata con CREATE MATERIALIZED VIEW, in modo da ridurre il tempo di risposta delle query di interesse da (a) a (c) sopra riportate. In particolare si specifichi la query SQL associata al Blocco A nella seguente istruzione:
CREATE MATERIALIZED VIEW ViewLezioni
BUILD IMMEDIATE
REFRESH FAST ON COMMIT
AS
	Blocco A

3. Si ipotizzi che la gestione della vista materializzata (tabella derivata) sia svolta mediante trigger. Scrivere il trigger per propagare le modifiche alla vista materializzata ViewLezioni in caso di inserimento di un nuovo record nella tabella dei fatti LEZIONI-TENNIS. 


In [ ]:
CREATE MATERIALIZED VIEW ViewLezioni
BUILD IMMEDIATE
REFRESH FAST ON COMMIT
AS
SELECT TD."2-Mesi", TD."3-Mesi", TD.Anno, C.Città, C.Regione, C.Provincia, C.Docce, C.Spogliatoi, C.Bar,
       JL.ModalitàPagamento, JL.TipologiaLezione, JL.LivelloLezione, 
       SUM(LT.Importo) AS ImportoTot,
       SUM(LT.NumAllievi) AS NumAllievi,
       SUM(LT.NumOre) AS NumOre
FROM "LEZIONI-TENNIS" LT
JOIN "TEMPO-DATA" TD ON TD.IDTempoD = LT.IDTempoD
JOIN CIRCOLO C ON C.IDCircolo = LT.IDCircolo
JOIN "JUNK-LEZIONE" JL ON JL.IDJL = LT.IDJL
GROUP BY TD."2-Mesi", TD."3-Mesi", TD.Anno, C.Città, C.Regione, C.Provincia, C.Docce, C.Spogliatoi, C.Bar,
         JL.ModalitàPagamento, JL.TipologiaLezione, JL.LivelloLezione ; 

2. Definire l’insieme minimale di attributi che permette di identificare le tuple appartenenti alla vista materializzata ViewLezioni.

Insieme Minimale = {2-Mesi, 3-Mesi, Città, Docce, Spogliatoi, Bar, ModalitàPagamento, TipologiaLezione, LivelloLezione}

3. Si ipotizzi che la gestione della vista materializzata (tabella derivata) sia svolta mediante trigger. Scrivere il trigger per propagare le modifiche alla vista materializzata ViewLezioni in caso di inserimento di un nuovo record nella tabella dei fatti LEZIONI-TENNIS. 

In [ ]:
CREATE OR REPLACE TRIGGER AggiornaViewLezioni
AFTER INSERT ON "LEZIONI-TENNIS"
FOR EACH ROW
DECLARE
    var2Mesi        "TEMPO-DATA"."2-Mesi"%TYPE;
    var3Mesi        "TEMPO-DAT"."3-Mesi"%TYPE;
    varAnno         "TEMPO-DATA".Anno%TYPE;

    varCittà, varRegione, varProvincia VARCHAR(20);
    varDocce, varSpogliatoi, varBar BOOLEAN;

    varModalitàPagamento, varTipologiaLezione, varLivelloLezione    VARCHAR(20);

    N INTEGER;

BEGIN
    SELECT "2-Mesi", "3-Mesi", Anno
    FROM "TEMPO-DATA"
    WHERE IDTempo = :NEW.IDTempo

    SELECT Città, Regione, PRovincia, Docce, Spogliotoi, Bar
    FROM CIRCOLO 
    WHERE IDCircolo = :NEW.IDCircolo

    SELECT ModalitàPagamento, TipologiaLezione, LivelloLezione
    FROM "JUNK-LEZIONI"
    WHERE IDJL = :NEW.IDJL

    SELECT COUNT (*)
    INTO N 
    FROM ViewLezioni 
    WHERE "2-Mesi" = var2Mesi
        AND "3-Mesi" = var3Mesi
        AND Anno = varAnno
        AND Città = varCittà
        AND Regione = varRegione
        AND Provincia = varProvincia
        AND Docce = varDocce
        AND Spogliatoi = varSpogliatoi
        AND Bar = varBar
        AND ModalitàPagamento = varModalitàPagamento
        AND TipologiaLezione = varTipologiaLezione
        AND LivelloLezione = varLivelloLezione
    
    IF (N>0) THEN
        UPDATE ViewLezioni
        SET ImportoTot = ImportoTot + :NEW.ImportoTot,
            NumAllievi = NumAllievi + :NEW.NumAllievi,
            NumOre = NumOre + :NEW.NumOre
        WHERE "2-Mesi" = var2Mesi
            AND "3-Mesi" = var3Mesi
            AND Città = varCittà
            AND Docce = varDocce
            AND Spogliatoi = varSpogliatoi
            AND Bar = varBar
            AND ModalitàPagamento = varModalitàPagamento
            AND TipologiaLezione = varTipologiaLezione
            AND LivelloLezione = varLivelloLezione
    ELSE 
        INSERT INTO ViewLezioni(
            "2-Mesi", "3-Mesi", Anno, Città, Regione, Provincia, Docce, Spogliatoi, Bar, 
            ModalitàPagamento, TipologiaLezione, LivelloLezione,
            ImportoTot, NumAllievi, NumOre
        )
        VALUES(
            var2Mesi, var3Mesi, varAnno, varCittà, varRegione, varProvincia, varDocce, varSpogliatoi, varBar,
            varModalitàPagamento, varTipologiaLezione, varLivelloLezione,
            :NEW.ImportoTot, :NEW.NumAllievi, :NEW.NumOre
        );
    END IF;
END;

